**FinBERT + BiLSTM**

**Architecture :**
```
Tweet → FinBERT → hidden states [128 × 768] → BiLSTM(256) → [512] → Dropout → Dense(128) → Dense(3)
```

**Imports & Configuration**

In [9]:
import os, json, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import AutoModel, AutoTokenizer,get_linear_schedule_with_warmup

from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_auc_score
)

# ── Seeds reproductibilité ────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# ── Device ───────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

# ── Chemins ──────────────────────────────────────────────────────────────────
DATA_DIR  = Path('data/processed')
RES_DIR   = Path('results')
MODEL_DIR = Path('models/fintwit_bilstm')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ── Hyperparamètres ──────────────────────────────────────────────────────────
BACKBONE     = 'ProsusAI/finbert'   # même tokenizer que le preprocessing (attention on a ajouter des tokens speciaux)
tokenizer = AutoTokenizer.from_pretrained(BACKBONE)

special_tokens = [f'TICKER_{s}' for s in [
    'AAPL','MSFT','GOOGL','AMZN','TSLA','META','NVDA','BRK','JPM',
    'JNJ','V','PG','UNH','HD','MA','PYPL','BAC','DIS','ADBE','NFLX',
    'CMCSA','XOM','VZ','INTC','T','CSCO','PFE'
]] + ['AMOUNT_']

tokenizer.add_tokens(special_tokens)
MAX_LENGTH   = 128
BATCH_SIZE   = 16     # réduit car on passe 128×768 hidden states (plus lourd que CLS seul)
GRAD_ACCUM   = 2      # gradient accumulation pour simuler batch_size=32
LR_BERT      = 2e-5
LR_HEAD      = 1e-3
WEIGHT_DECAY = 0.01
MAX_EPOCHS   = 10
PATIENCE     = 3
WARMUP_RATIO = 0.10

# Couches FinBERT à geler (les 8 premières sur 12)
# Justification : on passe les hidden states complets → modèle plus grand → risque overfitting plus élevé
# On gèle les couches basses (syntaxe générale) et dégèle les hautes (sémantique financière)
FREEZE_LAYERS = 8

LABEL_NAMES = ['négatif', 'neutre', 'positif']


Device : cpu


**Chargement des données**

In [2]:
train_enc = torch.load(DATA_DIR / 'train_encodings.pt')
val_enc   = torch.load(DATA_DIR / 'val_encodings.pt')
test_enc  = torch.load(DATA_DIR / 'test_encodings.pt')

print(f'Train  : {train_enc["input_ids"].shape[0]} tweets')
print(f'Val    : {val_enc["input_ids"].shape[0]} tweets')
print(f'Test   : {test_enc["input_ids"].shape[0]} tweets')

C:\Users\cherb\AppData\Local\Temp\ipykernel_40804\1672103546.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train_enc = torch.load(DATA_DIR / 'train_encodings.pt')


Train  : 29576 tweets
Val    : 6338 tweets
Test   : 6338 tweets


C:\Users\cherb\AppData\Local\Temp\ipykernel_40804\1672103546.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  val_enc   = torch.load(DATA_DIR / 'val_encodings.pt')
C:\Use

In [3]:
class TweetDataset(Dataset):
    def __init__(self, encodings):
        self.input_ids      = encodings['input_ids']
        self.attention_mask = encodings['attention_mask']
        self.labels         = encodings['labels']

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels':         self.labels[idx]
        }

train_loader = DataLoader(TweetDataset(train_enc), batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(TweetDataset(val_enc),   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(TweetDataset(test_enc),  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f'Batches — Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

Batches — Train: 1849 | Val: 397 | Test: 397


**Class weights**

In [4]:
with open(RES_DIR / 'class_weights.json') as f:
    cw_data = json.load(f)

cw = cw_data['weights']
class_weights_tensor = torch.tensor(
    [cw['0'], cw['1'], cw['2']], dtype=torch.float
).to(DEVICE)

print('Class weights :', {LABEL_NAMES[i]: round(cw[str(i)], 3) for i in range(3)})

Class weights : {'négatif': 1.765, 'neutre': 0.763, 'positif': 0.891}


**Rappel des métriques précedentes**

In [ ]:
with open(RES_DIR / 'metrics_baseline.json') as f:
    baseline = json.load(f)

with open(RES_DIR / 'metrics_01.json') as f:
    metrics_01 = json.load(f)

print('=== RÉFÉRENCES À BATTRE ===')
print(f'  Baseline TF-IDF  F1-macro : {baseline["f1_macro"]:.4f}')
print(f'  01 FinBERT seul F1-macro : {metrics_01["f1_macro"]:.4f}')
print()
print('  F1 par classe — 01 :')
for cls in ['negatif', 'neutre', 'positif']:
    print(f'    {cls:8s} : {metrics_01["f1_per_class"][cls]:.4f}')

**Architecture du modèle**

**Pourquoi les hidden states complets et pas juste le CLS ?**

Le token CLS résume le tweet en un seul vecteur — c'est une compression.  
Les 128 hidden states conservent la représentation de chaque token dans son contexte.  
Le BiLSTM peut alors exploiter l'**ordre** et les **dépendances locales** :  
- Négations : *"not bullish"* → le BiLSTM voit *not* juste avant *bullish*  
- Renversements : *"was strong but now dumping"* → le BiLSTM suit la progression  
- Intensificateurs : *"extremely bearish"* → le BiLSTM pondère *extremely* avant *bearish*

**Pourquoi geler les 8 premières couches ?**  
Les premières couches de BERT capturent la syntaxe générale (déjà bien apprise).  
Les 4 dernières capturent la sémantique spécifique → on les laisse s'adapter.  
Avec les hidden states complets le modèle est plus grand → plus de risque d'overfitting.

In [10]:
class FinBertBiLSTM(nn.Module):
    """
    FinBERT + BiLSTM + Classification Head.

    Flux :
      input_ids, attention_mask
        → FinBERT encoder → hidden states [batch, 128, 768]
        → BiLSTM(hidden=256, bidirectionnel)  
           → dernier état caché : [batch, 512]  (256 forward + 256 backward)
        → Dropout(0.3)
        → Linear(512 → 128) + ReLU
        → Dropout(0.2)
        → Linear(128 → 3)
    """
    def __init__(self, backbone_name, num_classes=3, lstm_hidden=256,
                 dropout1=0.3, dropout2=0.2, freeze_layers=8):
        super().__init__()

        # Backbone
        self.bert = AutoModel.from_pretrained(backbone_name)
        hidden_size = self.bert.config.hidden_size  # 768

        # Geler les N premières couches du transformer
        self._freeze_bert_layers(freeze_layers)

        # BiLSTM — bidirectionnel donc la sortie fait lstm_hidden * 2
        self.bilstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # Tête de classification
        self.drop1      = nn.Dropout(dropout1)
        self.dense      = nn.Linear(lstm_hidden * 2, 128)  # 512 → 128
        self.relu       = nn.ReLU()
        self.drop2      = nn.Dropout(dropout2)
        self.classifier = nn.Linear(128, num_classes)

    def _freeze_bert_layers(self, n_layers):
        """Gèle les embeddings + les n_layers premières couches transformer."""
        # Geler les embeddings
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False

        # Geler les n premières couches encoder
        for i, layer in enumerate(self.bert.encoder.layer):
            if i < n_layers:
                for param in layer.parameters():
                    param.requires_grad = False

        frozen = sum(1 for p in self.bert.parameters() if not p.requires_grad)
        total  = sum(1 for p in self.bert.parameters())
        print(f'  Couches BERT gelées : {n_layers}/12')
        print(f'  Params BERT gelés   : {frozen}/{total}')

    def forward(self, input_ids, attention_mask):
        # FinBERT → tous les hidden states de la séquence
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state  # shape : (batch, 128, 768)

        # BiLSTM traite la séquence complète
        # lstm_out shape : (batch, 128, 512)  — return_sequences=True implicite
        # On prend le DERNIER pas de temps (résumé de toute la séquence)
        lstm_out, _ = self.bilstm(hidden_states)
        last_hidden  = lstm_out[:, -1, :]  # shape : (batch, 512)

        # Tête de classification
        x = self.drop1(last_hidden)
        x = self.relu(self.dense(x))
        x = self.drop2(x)
        logits = self.classifier(x)  # shape : (batch, 3)

        return logits


# ── Instanciation ─────────────────────────────────────────────────────────────
model = FinBertBiLSTM(BACKBONE, freeze_layers=FREEZE_LAYERS).to(DEVICE)
model.bert.resize_token_embeddings(len(tokenizer))


trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params     = sum(p.numel() for p in model.parameters())
print(f'\nParamètres entraînables : {trainable_params:,} / {total_params:,}')

  Couches BERT gelées : 8/12
  Params BERT gelés   : 133/199


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`



Paramètres entraînables : 31,109,379 / 111,671,043


**Optimizer, Scheduler, Loss**

In [11]:
# ── Learning rate différencié (même logique que modele 01) ─────────────────────────
bert_params = [p for p in model.bert.parameters() if p.requires_grad]
head_params = [
    *model.bilstm.parameters(),
    *model.dense.parameters(),
    *model.classifier.parameters()
]

optimizer = AdamW([
    {'params': bert_params, 'lr': LR_BERT},
    {'params': head_params, 'lr': LR_HEAD}
], weight_decay=WEIGHT_DECAY)

# Steps effectifs = steps / grad_accum
effective_steps = (len(train_loader) // GRAD_ACCUM) * MAX_EPOCHS
warmup_steps    = int(effective_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=effective_steps
)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

print(f'Steps effectifs : {effective_steps} | Warmup : {warmup_steps}')
print(f'Gradient accumulation : {GRAD_ACCUM} (batch effectif = {BATCH_SIZE * GRAD_ACCUM})')

Steps effectifs : 9240 | Warmup : 924
Gradient accumulation : 2 (batch effectif = 32)


**Entrainement**

In [12]:
def train_epoch(model, loader, optimizer, scheduler, criterion, grad_accum):
    """Une époque avec gradient accumulation."""
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for step, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)

        logits = model(input_ids, attention_mask)
        loss   = criterion(logits, labels) / grad_accum  # normaliser par l'accumulation
        loss.backward()

        total_loss += loss.item() * grad_accum  # reconstituer la loss réelle

        # Mettre à jour les poids tous les grad_accum steps
        if (step + 1) % grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    return total_loss / len(loader)


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['labels'].to(DEVICE)

            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, labels)
            probs  = torch.softmax(logits, dim=-1)

            total_loss  += loss.item()
            all_preds   += logits.argmax(dim=-1).cpu().tolist()
            all_labels  += labels.cpu().tolist()
            all_probs   += probs.cpu().tolist()

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average='macro')

    return avg_loss, acc, f1_macro, all_preds, all_labels, all_probs

In [13]:
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}

best_val_f1  = 0.0
patience_cpt = 0
best_epoch   = 0
start_time   = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()

    train_loss                          = train_epoch(model, train_loader, optimizer, scheduler, criterion, GRAD_ACCUM)
    val_loss, val_acc, val_f1, _, _, _ = evaluate(model, val_loader, criterion)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    elapsed = time.time() - t0
    print(f'Epoch {epoch:02d}/{MAX_EPOCHS} | '
          f'Train Loss: {train_loss:.4f} | '
          f'Val Loss: {val_loss:.4f} | '
          f'Val Acc: {val_acc:.4f} | '
          f'Val F1: {val_f1:.4f} | '
          f'{elapsed:.0f}s')

    if val_f1 > best_val_f1:
        best_val_f1  = val_f1
        best_epoch   = epoch
        patience_cpt = 0
        torch.save(model.state_dict(), MODEL_DIR / 'best_model.pt')
        print(f'  → Meilleur modèle sauvegardé (F1-macro val = {val_f1:.4f})')
    else:
        patience_cpt += 1
        if patience_cpt >= PATIENCE:
            print(f'\nEarly stopping à l\'époque {epoch}')
            break

total_time = (time.time() - start_time) / 60
print(f'\nEntraînement terminé en {total_time:.1f} min')
print(f'Meilleure époque : {best_epoch} | Meilleur val F1-macro : {best_val_f1:.4f}')

KeyboardInterrupt: 

**Courbes d'apprentissage**

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_ran, history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(epochs_ran, history['val_loss'],   label='Val Loss',   marker='o')
axes[0].axvline(x=best_epoch, color='red', linestyle='--', label=f'Best epoch {best_epoch}')
axes[0].set_title('Loss')
axes[0].set_xlabel('Époque')
axes[0].legend()

axes[1].plot(epochs_ran, history['val_f1'], label='02 Val F1-macro', marker='o', color='blue')
axes[1].axhline(y=baseline['f1_macro'],      color='gray',  linestyle='--', label=f'Baseline ({baseline["f1_macro"]:.4f})')
axes[1].axhline(y=metrics_01['f1_macro'],   color='orange',linestyle='--', label=f'01 FinBERT ({metrics_01["f1_macro"]:.4f})')
axes[1].set_title('Val F1-macro vs références')
axes[1].set_xlabel('Époque')
axes[1].legend()

plt.tight_layout()
plt.savefig(RES_DIR / 'learning_curves_02.png', dpi=150)
plt.show()

**Évaluation finale sur le test set**

In [ ]:
model.load_state_dict(torch.load(MODEL_DIR / 'best_model.pt', map_location=DEVICE))

_, test_acc, test_f1, test_preds, test_labels, test_probs = evaluate(
    model, test_loader, criterion
)

f1_per_class = f1_score(test_labels, test_preds, average=None)
roc_auc      = roc_auc_score(test_labels, test_probs, multi_class='ovr', average='macro')

print('=== RÉSULTATS TEST — 02 FinBERT + BiLSTM ===')
print(f'  Accuracy  : {test_acc:.4f}')
print(f'  F1-macro  : {test_f1:.4f}')
print(f'  ROC-AUC   : {roc_auc:.4f}')
print(f'  F1 négatif : {f1_per_class[0]:.4f}')
print(f'  F1 neutre  : {f1_per_class[1]:.4f}')
print(f'  F1 positif : {f1_per_class[2]:.4f}')
print()
print(classification_report(test_labels, test_preds, target_names=LABEL_NAMES))

**Matrice de confusion**

In [ ]:
cm = confusion_matrix(test_labels, test_preds, normalize='true')

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='.2%', cmap='Blues',
    xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES
)
plt.title('Matrice de confusion — 02 FinBERT + BiLSTM (test, normalisée)')
plt.ylabel('Réel')
plt.xlabel('Prédit')
plt.tight_layout()
plt.savefig(RES_DIR / 'confusion_matrix_02.png', dpi=150)
plt.show()

**Sauvegarde des métriques**

In [ ]:
metrics_02 = {
    'model':                '02_finbert_bilstm',
    'accuracy':             round(test_acc, 4),
    'f1_macro':             round(test_f1, 4),
    'f1_per_class': {
        'negatif':          round(float(f1_per_class[0]), 4),
        'neutre':           round(float(f1_per_class[1]), 4),
        'positif':          round(float(f1_per_class[2]), 4),
    },
    'roc_auc':              round(roc_auc, 4),
    'training_time_minutes': round(total_time, 1),
    'best_epoch':           best_epoch,
    'trainable_params':     trainable_params,
    'val_f1_history':       [round(v, 4) for v in history['val_f1']],
    'frozen_bert_layers':   FREEZE_LAYERS,
    'grad_accum_steps':     GRAD_ACCUM,
}

with open(RES_DIR / 'metrics_02.json', 'w') as f:
    json.dump(metrics_02, f, indent=2)

print('Métriques sauvegardées : results/metrics_02.json')
print(json.dumps(metrics_02, indent=2))